In [1]:
import os
import numpy as np
import scipy.io
import matplotlib.pyplot as plt

cue_duration = 4.0
channels_plot = [0, 15, 30, 45, 58]

def load_mat(file_path: str):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Dataset not found")
    
    mat = scipy.io.loadmat(file_path, struct_as_record=False, squeeze_me=True)

    cnt = mat["cnt"]
    mrk = mat["mrk"]
    nfo = mat["nfo"]

    return cnt, mrk, nfo

def extract_trials(cnt: np.ndarray, mrk, fs: float, cue_duration: float):
    window_length = int(round(cue_duration * float(fs)))
    pos = np.array(mrk.pos, dtype=int).reshape(-1)
    y = np.array(mrk.y).reshape(-1)
    unique_labels = np.unique(y)

    # Mapping the labels to 0 and 1
    if set(unique_labels.tolist()) == {-1, 1}:
        y_mapped = (y == 1).astype(int)
        class_map = {-1: 0, 1: 1}
    elif set(unique_labels.tolist()) == {1, 2}:
        y_mapped = (y == 2).astype(int)
        class_map = {1: 0, 2: 1}
    else:
        y_mapped = y
        class_map = {int(k): int(k) for k in unique_labels}

    if cnt.ndim != 2:
        raise ValueError(f"Expected cnt to be 2D(samples, channels). Got shape {cnt.shape}")
    
    n_samples, n_channels = cnt.shape

    trials = []
    kept_labels = []

    for i, start in enumerate(pos):
        end = start + window_length
        if start < 0 or end > n_samples:
            continue
        seg = cnt[start:end, :]      # (samples, channels)
        trials.append(seg.T)         # (channels, samples)
        kept_labels.append(y_mapped[i])

    if not trials:
        raise RuntimeError("No trials were extracted, check your data format again")
    
    X = np.stack(trials, axis=0)  # trials, channels, samples
    y_out = np.array(kept_labels)

    return X, y_out, window_length, class_map

def train_test_split(X: np.ndarray, y: np.ndarray, train_ratio: float = 0.75, seed: int = 42):
    rng = np.random.default_rng(seed)
    idx = np.arange(X.shape[0])
    rng.shuffle(idx)

    n_train = int(round(train_ratio * len(idx)))
    train_idx = idx[:n_train]
    test_idx = idx[n_train:]

    return X[train_idx], y[train_idx], X[test_idx], y[test_idx]

def plot_sample(X: np.ndarray, fs: float, channels_plot: list[int], sample_index: int = 0):
    # X: (trials, channels, samples)
    x = X[sample_index]  # (channels, samples)
    n_ch, n_samp = x.shape
    t = np.arange(n_samp) / float(fs)

    plt.figure()
    for ch in channels_plot:  # <-- 0-based indices
        if ch < 0 or ch >= n_ch:
            continue
        plt.plot(t, x[ch], label=f"ch {ch}")

    plt.xlabel("Time(s)")
    plt.ylabel("Amplitude")
    plt.title(f"one trial (index {sample_index}) - selected channels")
    plt.legend()
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    file_path = os.path.join("data", "raw", "BCICIV_calib_ds1a.mat")
    cnt, mrk, nfo = load_mat(file_path)

    fs = float(nfo.fs)
    print("fs:", fs)
    print("cnt shape:", cnt.shape)

    X, y, window_length, class_map = extract_trials(cnt, mrk, fs, cue_duration)

    print("window_length (samples):", window_length)
    print("X shape:", X.shape)  # (trials, channels, samples)
    print("y shape:", y.shape)
    print("class_map:", class_map)

    X_train, y_train, X_test, y_test = train_test_split(X, y, train_ratio=0.75, seed=42)
    print("Train shapes:", X_train.shape, y_train.shape)
    print("Test shapes:", X_test.shape, y_test.shape)

    plot_sample(X_train, fs, channels_plot, sample_index=0)

    os.makedirs(os.path.join("data", "processed"), exist_ok=True)
    np.savez(
        os.path.join("data", "processed", "ds1a_extracted.npz"),
        X=X, y=y,
        X_train=X_train, y_train=y_train,
        X_test=X_test, y_test=y_test,
        fs=fs, window_length=window_length,
        class_map=class_map
    )


FileNotFoundError: Dataset not found

In [ ]:
import os
import numpy as np
from scipy.signal import butter, filtfilt

def bandpass_filter(X, fs, lowf, highf, order = 4):
    nyq = 0.5 * fs
    low = lowf / nyq
    high = highf / nyq

    b, a = butter(order, [low,high], btype = "bandpass")
    X_filt = np.zeros_like(X, dtype = np.float64)

    for i in range(X.shape[0]):
        for ch in range(X.shape[1]):
            X_filt[i, ch] = filtfilt(b, a, X[i, ch])

    return X_filt

def main():
    in_path = os.path.join("data", "processed", "ds1a_extracted.npz")
    out_path = os.path.join("data", "processed", "ds1a_filtered_bands.npz")

    data = np.load(in_path, allow_pickle = True)

    X = data["X"]
    y = data["y"]
    fs = float(data["fs"])
    print("Loaded dataset:", X.shape)
    
    print("Filtering 8-30 Hz...")
    X_8_30 = bandpass_filter(X, fs, 8, 30)

    print("Filering 8-13 Hz...")
    X_mu = bandpass_filter(X, fs, 8, 13)

    print("Filtering 13-30 Hz...")
    X_beta = bandpass_filter(X, fs, 13, 30)

    np.savez(
        out_path,
        X_8_30 = X_8_30,
        X_mu = X_mu,
        X_beta = X_beta,
        y = y,
        fs = fs

    )

    print("Saved filtered datasets to:", out_path)

if __name__ == "__main__":
        main()

In [ ]:
import numpy as np

data = np.load("data/processed/ds1a_extracted.npz")

print("Keys inside file:")
print(data.files)

print("\nX shape:", data["X"].shape)
print("y shape:", data["y"].shape)

print("\nTrain shape:", data["X_train"].shape)
print("Test shape:", data["X_test"].shape)


In [ ]:
import os 
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

BAND_KEY = "X_8_30"


N_PAIRS = 6
REG = 1e-10
SEED = 42

def cov_trace_norm(trial):
    c = trial @ trial.T
    tr = np.trace(c)
    return c / tr if tr > 0 else c

def mean_cov(X):
    return np.mean([cov_trace_norm(tr) for tr in X], axis = 0)

def whitener(C, eps = 1e-12):
    evals, evecs = np.linalg.eigh(C)
    evals = np.maximum(evals, eps)
    D_inv_sqrt = np.diag(1.0/np.sqrt(evals))
    P = D_inv_sqrt @ evecs.T
    return P

def fit_csp(X, y, n_pairs = 4, reg = 1e-10):
    X0 = X[y == 0]
    X1 = X[y == 1]
    if len(X0) == 0 or len(X1) == 0:
        raise ValueError("Need trials from both classes to fit CSP")
    
    C0 = mean_cov(X0) + reg * np.eye(X.shape[1])
    C1 = mean_cov(X1) + reg * np.eye(X.shape[1])
    C = C0 + C1

    P = whitener(C)
    S0 = P @ C0 @ P.T

    evals, B = np.linalg.eigh(S0)

    idx = np.argsort(evals)
    B = B[:, idx]

    W_full = (B.T @ P)
    
    picks = np.r_[0:n_pairs, -n_pairs:0]
    W = W_full[picks, :].T

    return W

def transform_csp(X, W):
    F = np.zeros((X.shape[0], W.shape[1]), dtype = np.float64)
    for i in range(X.shape[0]):
        Z = W.T  @ X[i]
        var = np.var(Z, axis = 1)
        F[i] = np.log(var + 1e-12)
    return F

def train_test_split_idx(n, train_ratio=0.75, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    n_train = int(round(train_ratio * n))
    return idx[:n_train], idx[n_train:]


def tsne_2d(features, seed=42):
    return TSNE(
        n_components=2,
        random_state=seed,
        init="pca",
        learning_rate="auto"
    ).fit_transform(features)


def plot_tsne(Z, y, title):
    plt.figure()
    plt.scatter(Z[y == 0, 0], Z[y == 0, 1], alpha=0.7, label="class 0")
    plt.scatter(Z[y == 1, 0], Z[y == 1, 1], alpha=0.7, label="class 1")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def main():
    # Load filtered data
    in_path = os.path.join("data", "processed", "ds1a_filtered_bands.npz")
    d = np.load(in_path, allow_pickle=True)

    X = d[BAND_KEY]            # (trials, channels, samples)
    X_mu = d["X_mu"]
    X_beta = d["X_beta"]
    y = d["y"].astype(int)     # (trials,)
    
    print("Loaded:", BAND_KEY, X.shape, "y:", y.shape)
    print("Loaded: X_mu", X_mu.shape, "X_beta", X_beta.shape)

    # Split first to avoid leakage
    train_idx, test_idx = train_test_split_idx(len(y), train_ratio=0.75, seed=SEED)
    X_train, y_train = X[train_idx], y[train_idx]
    X_test, y_test = X[test_idx], y[test_idx]
    X_mu_train = X_mu[train_idx]
    X_mu_test = X_mu[test_idx]
    X_beta_train = X_beta[train_idx]
    X_beta_test = X_beta[test_idx]

    # ---------------- t-SNE BEFORE CSP ----------------

    F_before = np.log(np.var(X_train, axis=2) + 1e-12)
    Z_before = tsne_2d(F_before, seed=SEED)
    plot_tsne(Z_before, y_train, f"t-SNE BEFORE CSP ({BAND_KEY}: per-channel log-variance)")

    # ---------------- CSP on 8-30  ----------------
    W = fit_csp(X_train, y_train, n_pairs=N_PAIRS, reg=REG)
    F_train = transform_csp(X_train, W)
    F_test = transform_csp(X_test, W)

    print("CSP 8-30 features:")
    print("F_train:", F_train.shape, "F_test:", F_test.shape)
    #----------------- CSP on mu and beta seperate + concatenate ---------
    W_mu = fit_csp(X_mu_train, y_train, n_pairs=N_PAIRS, reg=REG)
    W_beta = fit_csp(X_beta_train, y_train, n_pairs=N_PAIRS, reg=REG)

    F_mu_train = transform_csp(X_mu_train, W_mu)
    F_mu_test = transform_csp(X_mu_test, W_mu)

    F_beta_train = transform_csp(X_beta_train, W_beta)
    F_beta_test = transform_csp(X_beta_test, W_beta)

    F_mubeta_train = np.concatenate([F_mu_train, F_beta_train], axis=1)
    F_mubeta_test = np.concatenate([F_mu_test, F_beta_test], axis=1)

    print("CSP Mu+Beta concatenated features:")
    print("F_mubeta_train:", F_mubeta_train.shape, "F_mubeta_test:", F_mubeta_test.shape)

    Z_mubeta = tsne_2d(F_mubeta_train, seed=SEED)
    plot_tsne(Z_mubeta, y_train, "t-SNE AFTER CSP (Mu CSP + Beta CSP concatenated)")
    # ---------------- t-SNE AFTER CSP ----------------
    Z_after = tsne_2d(F_train, seed=SEED)
    plot_tsne(Z_after, y_train, f"t-SNE AFTER CSP ({BAND_KEY}: CSP log-variance features)")

    # Save CSP features for next step (classification)
    out_path = os.path.join("data", "processed", "ds1a_csp_features.npz")
    np.savez(
        out_path,
        band_key=BAND_KEY,
        train_idx=train_idx,
        test_idx=test_idx,
        y=y,
        F_train=F_train,
        y_train=y_train,
        F_test=F_test,
        y_test=y_test,
        F_mubeta_train=F_mubeta_train,
        F_mubeta_test=F_mubeta_test,
        n_pairs=N_PAIRS
    )
    print("Saved CSP features to:", out_path)


if __name__ == "__main__":
    main()

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Optional, Tuple, Dict
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
    RocCurveDisplay,
    classification_report
)


# --------------------------
# Utils: Standardization
# --------------------------
@dataclass
class Standardizer:
    mean_: Optional[np.ndarray] = None
    std_: Optional[np.ndarray] = None

    def fit(self, X: np.ndarray):
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        self.std_[self.std_ == 0] = 1.0
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        return (X - self.mean_) / self.std_

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)


# --------------------------
# RBF Kernel
# --------------------------
def rbf_kernel(X1: np.ndarray, X2: np.ndarray, gamma: float) -> np.ndarray:
    # K(x,z) = exp(-gamma * ||x-z||^2)
    # Efficient pairwise squared distances:
    X1_sq = np.sum(X1 * X1, axis=1, keepdims=True)      # (n1,1)
    X2_sq = np.sum(X2 * X2, axis=1, keepdims=True).T    # (1,n2)
    dist2 = X1_sq + X2_sq - 2.0 * (X1 @ X2.T)
    return np.exp(-gamma * dist2)


# --------------------------
# SVM (RBF) from scratch using SMO
# Binary labels must be in {-1, +1}
# --------------------------
@dataclass
class SVMRBF_SM0:
    C: float = 1.0
    gamma: float = 0.1
    tol: float = 1e-3
    max_passes: int = 10
    max_iters: int = 20000
    seed: int = 42

    # learned params
    alphas_: Optional[np.ndarray] = None
    b_: float = 0.0
    X_sv_: Optional[np.ndarray] = None
    y_sv_: Optional[np.ndarray] = None
    alpha_sv_: Optional[np.ndarray] = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        rng = np.random.default_rng(self.seed)
        n = X.shape[0]
        y = y.astype(float)

        # precompute full kernel matrix
        K = rbf_kernel(X, X, self.gamma)

        alphas = np.zeros(n, dtype=float)
        b = 0.0

        def f(i):
            return np.sum(alphas * y * K[:, i]) + b

        passes = 0
        iters = 0

        while passes < self.max_passes and iters < self.max_iters:
            num_changed = 0
            for i in range(n):
                Ei = f(i) - y[i]

                # check KKT violation
                if (y[i] * Ei < -self.tol and alphas[i] < self.C) or (y[i] * Ei > self.tol and alphas[i] > 0):
                    # pick j != i randomly
                    j = i
                    while j == i:
                        j = rng.integers(0, n)
                    Ej = f(j) - y[j]

                    ai_old = alphas[i]
                    aj_old = alphas[j]

                    # bounds L, H
                    if y[i] != y[j]:
                        L = max(0.0, aj_old - ai_old)
                        H = min(self.C, self.C + aj_old - ai_old)
                    else:
                        L = max(0.0, ai_old + aj_old - self.C)
                        H = min(self.C, ai_old + aj_old)

                    if abs(H - L) < 1e-12:
                        continue

                    # eta = 2Kij - Kii - Kjj
                    eta = 2.0 * K[i, j] - K[i, i] - K[j, j]
                    if eta >= 0:
                        continue

                    # update aj
                    aj_new = aj_old - (y[j] * (Ei - Ej)) / eta
                    # clip
                    aj_new = np.clip(aj_new, L, H)

                    if abs(aj_new - aj_old) < 1e-5:
                        continue

                    # update ai
                    ai_new = ai_old + y[i] * y[j] * (aj_old - aj_new)

                    # update b
                    b1 = b - Ei - y[i] * (ai_new - ai_old) * K[i, i] - y[j] * (aj_new - aj_old) * K[i, j]
                    b2 = b - Ej - y[i] * (ai_new - ai_old) * K[i, j] - y[j] * (aj_new - aj_old) * K[j, j]

                    # choose b
                    if 0 < ai_new < self.C:
                        b = b1
                    elif 0 < aj_new < self.C:
                        b = b2
                    else:
                        b = 0.5 * (b1 + b2)

                    alphas[i] = ai_new
                    alphas[j] = aj_new
                    num_changed += 1

                iters += 1
                if iters >= self.max_iters:
                    break

            if num_changed == 0:
                passes += 1
            else:
                passes = 0

        # keep support vectors
        sv_mask = alphas > 1e-8
        self.alphas_ = alphas
        self.b_ = b
        self.X_sv_ = X[sv_mask]
        self.y_sv_ = y[sv_mask]
        self.alpha_sv_ = alphas[sv_mask]
        return self

    def decision_function(self, X: np.ndarray) -> np.ndarray:
        K = rbf_kernel(self.X_sv_, X, self.gamma)  # (n_sv, n)
        scores = (self.alpha_sv_ * self.y_sv_) @ K + self.b_
        return scores

    def predict(self, X: np.ndarray) -> np.ndarray:
        scores = self.decision_function(X)
        return (scores >= 0).astype(int)  # map to {0,1}


# --------------------------
# Evaluation + plots
# --------------------------
def evaluate_binary(y_true01: np.ndarray, y_pred01: np.ndarray, scores: np.ndarray, title_prefix: str):
    acc = accuracy_score(y_true01, y_pred01)
    prec = precision_score(y_true01, y_pred01, zero_division=0)
    rec = recall_score(y_true01, y_pred01, zero_division=0)
    f1 = f1_score(y_true01, y_pred01, zero_division=0)

    print(f"\n[{title_prefix}] Metrics")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1       : {f1:.4f}")
    print("\nClassification report:")
    print(classification_report(y_true01, y_pred01, digits=4, zero_division=0))

    cm = confusion_matrix(y_true01, y_pred01)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f"{title_prefix} - Confusion Matrix")
    plt.tight_layout()
    plt.show()

    fpr, tpr, _ = roc_curve(y_true01, scores)
    roc_auc = auc(fpr, tpr)
    RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc).plot()
    plt.title(f"{title_prefix} - ROC (AUC={roc_auc:.4f})")
    plt.tight_layout()
    plt.show()

    return {"acc": acc, "prec": prec, "rec": rec, "f1": f1, "auc": roc_auc}


# --------------------------
# Grid search (simple) on train only
# --------------------------
def simple_grid_search_custom_svm(
    Xtr: np.ndarray, ytr01: np.ndarray,
    C_list=(0.1, 1.0, 10.0, 100.0),
    gamma_list=(0.01, 0.1, 1.0, 10.0),
    seed=42
) -> Tuple[float, float, Dict]:
    # use a small internal split inside train for model selection (holdout)
    rng = np.random.default_rng(seed)
    idx = np.arange(len(ytr01))
    rng.shuffle(idx)
    split = int(round(0.8 * len(idx)))
    fit_idx, val_idx = idx[:split], idx[split:]

    X_fit, y_fit = Xtr[fit_idx], ytr01[fit_idx]
    X_val, y_val = Xtr[val_idx], ytr01[val_idx]

    y_fit_pm = np.where(y_fit == 1, 1.0, -1.0)

    best = {"acc": -1.0, "C": None, "gamma": None}
    for C in C_list:
        for gamma in gamma_list:
            model = SVMRBF_SM0(C=C, gamma=gamma, seed=seed, max_passes=10, max_iters=20000)
            model.fit(X_fit, y_fit_pm)
            val_scores = model.decision_function(X_val)
            val_pred = (val_scores >= 0).astype(int)
            acc = accuracy_score(y_val, val_pred)
            if acc > best["acc"]:
                best.update({"acc": acc, "C": C, "gamma": gamma})

    return best["C"], best["gamma"], best


# --------------------------
# Run for one feature set
# --------------------------
def run_one_branch(name: str, X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray):
    print("\n" + "=" * 80)
    print(f"BRANCH: {name}")
    print("=" * 80)

    # 1) Standardize (fit on train only)
    scaler = Standardizer()
    Xtr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)

    # 2) Hyperparam selection on train only (simple holdout inside train)
    C_best, gamma_best, best_info = simple_grid_search_custom_svm(Xtr, y_train, seed=42)
    print(f"\n[{name}] Best (custom grid): C={C_best}, gamma={gamma_best}, val_acc={best_info['acc']:.4f}")

    # 3) Train CUSTOM SVM from scratch on full train
    ytr_pm = np.where(y_train == 1, 1.0, -1.0)
    custom = SVMRBF_SM0(C=C_best, gamma=gamma_best, seed=42, max_passes=15, max_iters=50000)
    custom.fit(Xtr, ytr_pm)

    custom_scores = custom.decision_function(Xte)          # real-valued scores
    custom_pred = (custom_scores >= 0).astype(int)

    custom_metrics = evaluate_binary(y_test, custom_pred, custom_scores, f"{name} | Custom RBF-SVM (SMO)")

    # 4) Train sklearn SVC with SAME params and compare
    sk = SVC(kernel="rbf", C=C_best, gamma=gamma_best)
    sk.fit(Xtr, y_train)

    sk_scores = sk.decision_function(Xte)
    sk_pred = sk.predict(Xte).astype(int)

    sk_metrics = evaluate_binary(y_test, sk_pred, sk_scores, f"{name} | scikit-learn SVC(RBF)")

    # 5) Compare side-by-side summary
    print(f"\n[{name}] SUMMARY COMPARISON (Test)")
    for k in ["acc", "prec", "rec", "f1", "auc"]:
        print(f"{k.upper():>4} | custom={custom_metrics[k]:.4f}   sklearn={sk_metrics[k]:.4f}")

    return {
        "best_C": C_best,
        "best_gamma": gamma_best,
        "custom": custom_metrics,
        "sklearn": sk_metrics
    }


def main():
    # load your saved features
    path = os.path.join("data", "processed", "ds1a_csp_features.npz")
    d = np.load(path, allow_pickle=True)

    # A) 8-30 CSP features
    F_train = d["F_train"]
    y_train = d["y_train"].astype(int)
    F_test = d["F_test"]
    y_test = d["y_test"].astype(int)

    # B) mu+beta concatenated CSP features
    F_mb_train = d["F_mubeta_train"]
    F_mb_test = d["F_mubeta_test"]

    print("Loaded features:")
    print("  F_train:", F_train.shape, "y_train:", y_train.shape)
    print("  F_test :", F_test.shape, "y_test :", y_test.shape)
    print("  F_mb_train:", F_mb_train.shape, "F_mb_test:", F_mb_test.shape)

    # Run BOTH branches exactly as you asked
    res_A = run_one_branch("A) CSP on 8-30Hz", F_train, y_train, F_test, y_test)
    res_B = run_one_branch("B) CSP on (mu+beta) concat", F_mb_train, y_train, F_mb_test, y_test)

    print("\n" + "#" * 80)
    print("FINAL RESULT (Test) - both branches")
    print("#" * 80)
    print("A) 8-30Hz  : custom acc={:.4f}, sklearn acc={:.4f}".format(res_A["custom"]["acc"], res_A["sklearn"]["acc"]))
    print("B) mu+beta : custom acc={:.4f}, sklearn acc={:.4f}".format(res_B["custom"]["acc"], res_B["sklearn"]["acc"]))
    print("#" * 80)


if __name__ == "__main__":
    main()


FileNotFoundError: [Errno 2] No such file or directory: 'data\\processed\\ds1a_csp_features.npz'